# YOLO11n N54 R1 — resume to epoch 100

Same QA-passed train/validation run, resumed from epoch 2 on a third NVIDIA L4. The sealed test stays closed.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

TRACK = 'n54'
MODE = 'resume'
STOP_AFTER = None
REPO_URL = 'https://github.com/ScottPhantom/Traffic_SIgn_Detection.git'
SOURCE_COMMIT = '45b7b1fab606032fcfb2956d0207171ae8b4c7f6'
SOURCE_QA_SHA256 = 'bcf7470d931d23445785c1103e3c919aca4c696b22738079756572892ce4f632'
WEIGHT_URL = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo11n.pt'
WEIGHT_SHA256 = '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1'
DRIVE_ROOT = Path('/content/drive/MyDrive/CPV301/Traffic_Sign_Detector')
STATE = DRIVE_ROOT / 'M1_D1_Colab/runs/yolo11n_n54_seed_20261018_r1'
REPO = Path('/content/Traffic_Sign_Detector')
assert MODE in {'fresh', 'resume'}
assert MODE == 'resume' and STOP_AFTER is None

N54_ROOT = DRIVE_ROOT / 'M1_D1_Colab'
DRIVE_BUNDLE = N54_ROOT / 'm1_colab_train_val_backend_r1.tar.gz'
SIDECAR = N54_ROOT / 'm1_colab_train_val_backend_r1.tar.gz.manifest.json'
BUNDLE_SHA256 = '7ea344cb1e2c166786d0552f215993969efec9b5f06a9bf49310d4390243d641'
BUNDLE_BYTES = 766226800
SIDECAR_SHA256 = '7e0dc5b0144faa5f0be407224eaf8070dd51199fa8a1d2f164b248fd6a06d5b2'
SIDECAR_BYTES = 1499866
PART_SPECS = (('m1_colab_train_val_backend_r1.tar.gz.part-00', 99614720, '53f30f08406fbef5b8451610b2fd44a1f2be42ca6d50f612b68e5ace285e66ff'), ('m1_colab_train_val_backend_r1.tar.gz.part-01', 99614720, '8451fe932760179d4b2e1a9cbdd21fd58744bc6b8f3d2a90e431cece7f546544'), ('m1_colab_train_val_backend_r1.tar.gz.part-02', 99614720, '867c935bf152ddfc3e7919547dccf3682291326b60b956a8100fc1a59b60e98d'), ('m1_colab_train_val_backend_r1.tar.gz.part-03', 99614720, '046fc8ae5d4e0a91196ccd6a6afa422fb1254baea7ae30bc3cab39602b534907'), ('m1_colab_train_val_backend_r1.tar.gz.part-04', 99614720, '6d3751d858232fada5d57e8e892f3e272cd5b9e34efb184971b3f4786842665d'), ('m1_colab_train_val_backend_r1.tar.gz.part-05', 99614720, '938106e4d34743e9b79109feb98d573f317afad44289f792f070f22e3924159b'), ('m1_colab_train_val_backend_r1.tar.gz.part-06', 99614720, '3497c1a10314c74d4b81ba9a5a49653b16ba2e2d52b424fd7655623547cb6408'), ('m1_colab_train_val_backend_r1.tar.gz.part-07', 68923760, '5f5ca85bc392310b2b5693ad9d0723f046eca9fd7980b45c944f22a3b3bc0eea'))
SOURCE_QA = REPO / 'docs/review/yolo11n_dual_track_r1_source_independent_qa_record.md'

SMOKE_QA = DRIVE_ROOT / 'M1_D1_Colab/yolo11n_n54_r1_smoke_qa_proof.json'
SMOKE_QA_SHA256 = '9189b52043157a5b536365fa714df8152bb05519b8bff723596f024a189bcd1e'
LATEST_SHA256 = '3e223f7cb010c83da168e239e1b1d9016129e44bffb2a36261b6234f9acc68a3'


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys

def file_hash(path):
    value = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()

gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert gpu_name == 'NVIDIA L4', gpu_name
assert len(SOURCE_COMMIT) == 40
assert len(SOURCE_QA_SHA256) == 64
assert len(WEIGHT_SHA256) == 64
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO, check=True)
observed = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
assert observed == SOURCE_COMMIT
assert not subprocess.run(
    ['git', 'status', '--porcelain'], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.11.14'], check=True)
print({'gpu': gpu_name, 'checkout': observed, 'track': TRACK, 'mode': MODE})


In [ ]:
import os
import urllib.request

assert SIDECAR.is_file() and not SIDECAR.is_symlink()
assert SIDECAR.stat().st_size == SIDECAR_BYTES
assert file_hash(SIDECAR) == SIDECAR_SHA256
if DRIVE_BUNDLE.is_file():
    assert not DRIVE_BUNDLE.is_symlink()
    BUNDLE = DRIVE_BUNDLE
    bundle_source = 'single_archive'
else:
    parts = []
    for name, expected_bytes, expected_hash in PART_SPECS:
        part = N54_ROOT / name
        assert part.is_file() and not part.is_symlink()
        assert part.stat().st_size == expected_bytes
        assert file_hash(part) == expected_hash
        parts.append(part)
    BUNDLE = Path('/content/m1_colab_train_val_backend_r1.tar.gz')
    if BUNDLE.exists():
        assert BUNDLE.is_file() and not BUNDLE.is_symlink()
    else:
        with BUNDLE.open('xb') as destination:
            for part in parts:
                with part.open('rb') as source:
                    for chunk in iter(lambda: source.read(1024 * 1024), b''):
                        destination.write(chunk)
    bundle_source = 'verified_drive_parts'
assert BUNDLE.stat().st_size == BUNDLE_BYTES
assert file_hash(BUNDLE) == BUNDLE_SHA256
assert SOURCE_QA.is_file() and file_hash(SOURCE_QA) == SOURCE_QA_SHA256
if MODE == 'fresh':
    assert not STATE.exists(), STATE
else:
    assert (STATE / 'recovery/latest.json').is_file()
weight = REPO / 'models/pretrained/yolo11n.pt'
weight.parent.mkdir(parents=True, exist_ok=True)
if weight.exists():
    assert weight.is_file() and not weight.is_symlink()
    assert file_hash(weight) == WEIGHT_SHA256
else:
    partial = weight.with_suffix('.partial')
    assert not partial.exists()
    urllib.request.urlretrieve(WEIGHT_URL, partial)
    assert file_hash(partial) == WEIGHT_SHA256
    os.replace(partial, weight)
assert file_hash(weight) == WEIGHT_SHA256
print({
    'bundle_source': bundle_source,
    'bundle_sha256': BUNDLE_SHA256,
    'weight_sha256': WEIGHT_SHA256,
})

assert SMOKE_QA.is_file() and not SMOKE_QA.is_symlink()
assert file_hash(SMOKE_QA) == SMOKE_QA_SHA256
latest_path = STATE / 'recovery/latest.json'
assert latest_path.is_file() and not latest_path.is_symlink()
assert file_hash(latest_path) == LATEST_SHA256
latest = json.loads(latest_path.read_text())
assert latest['generation'] == 'epoch-0002'
execution = json.loads((STATE / 'colab_execution.json').read_text())
assert execution['status'] == 'PAUSED_FOR_RESUME'
assert execution['sealed_test_accessed'] is False
assert [attempt['completed_epoch'] for attempt in execution['execution_attempts']] == [1, 2]
proof = json.loads(SMOKE_QA.read_text())
assert proof['verdict'] == 'PASS'
assert proof['checkpoint_latest_sha256'] == LATEST_SHA256
assert proof['sealed_test_accessed'] is False
assert proof['runtime_digests'] == [
    attempt['runtime_digest'] for attempt in execution['execution_attempts']
]
assert len(set(proof['runtime_digests'])) == 2
subprocess.run(['uv', 'sync', '--frozen'], cwd=REPO / 'detector', check=True)
print({'resume_from': 'epoch-0002', 'smoke_qa_sha256': SMOKE_QA_SHA256})


In [ ]:
command = [
    sys.executable, str(REPO / 'detector/colab/run_yolo11n_n54_r1.py'),
    '--repo-root', str(REPO),
    '--bundle', str(BUNDLE), '--bundle-sha256', BUNDLE_SHA256,
    '--state-dir', str(STATE), '--mode', MODE,
    '--expected-checkout', SOURCE_COMMIT,
    '--source-qa-record', str(SOURCE_QA), '--source-qa-sha256', SOURCE_QA_SHA256,
    '--smoke-qa-proof', str(SMOKE_QA), '--smoke-qa-sha256', SMOKE_QA_SHA256,
    '--seed', '20261018', '--epochs', '100', '--imgsz', '1280',
    '--batch', '4', '--workers', '4', '--fraction', '1.0',
    '--patience', '30', '--device', '0', '--tag', 'n54_r1', '--cos-lr',
]
assert '--stop-after-epoch' not in command
print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import sqlite3
import tarfile

execution = json.loads((STATE / 'colab_execution.json').read_text())
assert execution['status'] == 'SUCCEEDED'
assert execution['sealed_test_accessed'] is False
attempts = execution['execution_attempts']
assert len(attempts) == 3
assert [item['leg'] for item in attempts] == [
    'fresh_epoch_1', 'resume_epoch_2', 'long_resume',
]
assert len({item['runtime_digest'] for item in attempts}) == 3
assert all(item['sealed_test_accessed'] is False for item in attempts)
assert attempts[-1]['completed_epoch'] == 100

latest = json.loads((STATE / 'recovery/latest.json').read_text())
assert latest['generation'] == 'epoch-0100'
snapshot = STATE / 'recovery/snapshots/epoch-0100'
checkpoint_manifest_path = snapshot / 'checkpoint_manifest.json'
assert file_hash(checkpoint_manifest_path) == latest['manifest_sha256']
checkpoint_manifest = json.loads(checkpoint_manifest_path.read_text())
assert checkpoint_manifest['completed_epoch'] == 100

result_dir = STATE / 'results'
archive_path = result_dir / 'result.tar.gz'
manifest_path = result_dir / 'result_manifest.json'
assert archive_path.is_file() and not archive_path.is_symlink()
assert manifest_path.is_file() and not manifest_path.is_symlink()
result = json.loads(manifest_path.read_text())
assert result['package'] == 'YOLO11N-N54-R1'
assert result['completed_epoch'] == 100
assert result['sealed_test_accessed'] is False
assert file_hash(archive_path) == result['archive_sha256']
expected = result['members']
assert len(expected) == 8
with tarfile.open(archive_path, 'r:gz') as reader:
    members = reader.getmembers()
    assert len(members) == 8 and {member.name for member in members} == set(expected)
    for member in members:
        assert member.isfile()
        content = reader.extractfile(member).read()
        assert len(content) == expected[member.name]['bytes']
        assert hashlib.sha256(content).hexdigest() == expected[member.name]['sha256']

database = STATE / 'DB_logger.db'
text_log = STATE / 'colab_train.log'
assert database.is_file() and text_log.is_file()
with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as db:
    assert db.execute('PRAGMA integrity_check').fetchone() == ('ok',)
    assert db.execute('SELECT COUNT(*) FROM logs').fetchone()[0] > 0
assert 'Training end' in text_log.read_text()
print({
    'status': 'SUCCEEDED', 'epoch': 100,
    'archive_sha256': result['archive_sha256'], 'seal': False,
})
